# Transformation to SQL query

In [0]:
query = """
SELECT
    ROW_NUMBER() OVER (ORDER BY p.product_valid_from, p.product_key) AS product_id,
    p.product_key,
    p.product_name,
    p.product_cost,
    p.product_line,
    p.product_valid_from,
    p.category_id,
    c.category,
    c.subcategory,
    c.requires_maintenance
FROM silver.crm_products p
LEFT JOIN silver.erp_product_categories c
    ON p.category_id = c.category_id
WHERE p.product_valid_to IS NULL  
"""
df = spark.sql(query)

# Preview

In [0]:
df.limit(10).display()


# Duplicate product_id check

In [0]:
duplicate_product_id = (
    df.groupBy("product_id")
      .count()
      .filter("count > 1")
      .count()
)

print("Duplicate product_id:", duplicate_product_id)

if duplicate_product_id > 0:
    raise Exception("Duplicate product_id found!")



#Write it to Gold Table

In [0]:
df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("workspace.gold.dim_products")

## Sanity check

In [0]:
%sql
SELECT *
FROM gold.dim_products
LIMIT(20)


